# Train the baseline model

## Overall Workflow
1. Read packages from unknown_packages.txt(unknown group) and vulnerable_packages.txt(positive group) and combine them into one collection
2. Initialize a FeatureGenerator with system = "pypi"
3. Iterate through package + version:
    + Generate package features for the package + version using FeatureGenerator's function of get_package_metadata(no structural data)
    + Append the package's embeddings into the tensor/dataframe initialized earlier
4. Fit a clustering Model (K-means) on these package embeddings (Tune K use Elbow Method)
5. Evaluate the clustering result (best K)
    + Define `risk(cluster) = #vulnerable_package / size of cluster`
    + Rank clusters by riskscore
    + Get TopK cluster, for example(10%)
    + Calculate the recall and precision of TopK cluster:
        + Recall: How many known vulnerable package is in TopK cluster?
        + Precision: Out of the TopK cluster, how much of them are known vulnerable packages
6. Save the baseline model

In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
import joblib

from scripts.FeatureGenerator import FeatureGenerator

VULNERABLE_FILE  = '../data/vulnerable_packages.txt'
UNKNOWN_FILE     = '../data/unknown_packages.txt'
METADATA_CACHE   = '../data/package_metadata_cache.csv'   # shared across models
FEATURES_CACHE   = '../data/baseline_features.csv'        # labeled dataset for this model
MODEL_OUTPUT     = '../models/lib/baseline_kmeans.pkl'
SCALER_OUTPUT    = '../models/lib/baseline_scaler.pkl'
IMPUTER_OUTPUT   = '../models/lib/baseline_imputer.pkl'

PACKAGE_FEATURES = [
    'num_authors', 'num_maintainers', 'num_roles', 'has_organization',
    'has_license', 'yanked', 'has_sdist', 'has_sig',
    'num_dependencies', 'has_requires_python',
    'num_releases', 'days_since_first_release', 'days_since_last_release', 'dev_status_level',
    'description_length', 'num_classifiers', 'num_project_urls', 'has_homepage',
    'num_wheel_dists', 'total_dist_size_kb',
]

os.makedirs('../models/lib', exist_ok=True)
print('Setup complete.')

Setup complete.


## Step 1-2: Load packages and assign labels

In [2]:
def load_packages(filepath, label):
    entries = []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if line:
                pkg, ver = line.rsplit('@', 1)
                entries.append({'package': pkg, 'version': ver, 'label': label})
    return entries

vulnerable_entries = load_packages(VULNERABLE_FILE, label=1)
unknown_entries    = load_packages(UNKNOWN_FILE,    label=0)
all_entries        = vulnerable_entries + unknown_entries

print(f'Vulnerable : {len(vulnerable_entries)}')
print(f'Unknown    : {len(unknown_entries)}')
print(f'Total      : {len(all_entries)}')

Vulnerable : 5000
Unknown    : 5000
Total      : 10000


## Step 3-4: Feature extraction

Calls the PyPI API for each package. Results are cached to `baseline_features.csv` — re-run this cell only when you want to refresh the data. Otherwise skip to the **Load cache** cell below.

In [3]:
fg = FeatureGenerator(system='pypi', cache_path=METADATA_CACHE)

rows = []
for entry in tqdm(all_entries, desc='Extracting features'):
    pkg, ver, label = entry['package'], entry['version'], entry['label']
    try:
        metadata = fg.get_package_metadata(pkg, ver)
        if not metadata:
            continue
        row = {f: metadata.get(f, None) for f in PACKAGE_FEATURES}
        row['package'] = pkg
        row['version'] = ver
        row['label']   = label
        rows.append(row)
    except Exception:
        pass

features_df = pd.DataFrame(rows)
features_df.to_csv(FEATURES_CACHE, index=False)
print(f'Extracted {len(features_df)} rows  ({features_df["label"].sum():.0f} vulnerable).')
features_df.head()

[FeatureGenerator] Loaded 32729 cached entries from ../data/package_metadata_cache.csv


Extracting features: 100%|██████████| 10000/10000 [00:00<00:00, 919380.11it/s]

Extracted 10000 rows  (5000 vulnerable).


,num_authors,num_maintainers,num_roles,has_organization,has_license,yanked,has_sdist,has_sig,num_dependencies,has_requires_python,...,dev_status_level,description_length,num_classifiers,num_project_urls,has_homepage,num_wheel_dists,total_dist_size_kb,package,version,label
0,1,1,2,1,1,0,1,0,10,1,...,3,16944,15,3,1,1,156,starlite,1.5.4,1
1,1,0,8,1,1,0,1,0,34,0,...,0,1367,9,1,1,2,11239,ipython,4.1.0,1
2,1,0,5,1,1,0,1,0,0,0,...,3,4042,24,1,1,0,57,dash,1.3.0,1
3,1,0,3,0,1,0,1,0,27,1,...,3,1718,7,6,1,1,37935,homeassistant,2023.5.4,1
4,1,0,2,0,1,0,1,0,0,0,...,1,2084,12,1,1,0,19,uvicorn,0.3.13,1


In [4]:
# Load cache (run this instead of the extraction cell if features already exist)
features_df = pd.read_csv(FEATURES_CACHE)
print(f'Loaded {len(features_df)} rows  ({features_df["label"].sum():.0f} vulnerable).')
features_df.describe()

Loaded 10000 rows  (5000 vulnerable).


,num_authors,num_maintainers,num_roles,has_organization,has_license,yanked,has_sdist,has_sig,num_dependencies,has_requires_python,...,days_since_first_release,days_since_last_release,dev_status_level,description_length,num_classifiers,num_project_urls,has_homepage,num_wheel_dists,total_dist_size_kb,label
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.0,10000.000000,10000.000000,...,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,1.000000e+04,10000.000000
mean,0.902400,0.068700,1.892300,0.099200,0.763000,0.008500,0.933100,0.0,13.097000,0.500600,...,2713.027500,756.752500,1.115600,5140.532000,7.211900,1.671600,0.854300,1.396000,1.439372e+04,0.500000
std,0.638682,0.335547,1.933979,0.298945,0.425264,0.091807,0.249861,0.0,66.769857,0.500025,...,1858.196281,1077.917669,1.258092,10884.912674,6.084973,1.503655,0.352823,4.460849,1.376347e+05,0.500025
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,...,7.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000
25%,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.0,0.000000,0.000000,...,1147.000000,16.000000,0.000000,218.000000,3.000000,1.000000,1.000000,0.000000,1.500000e+01,0.000000
50%,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.0,1.000000,1.000000,...,2592.000000,203.000000,0.000000,2014.000000,7.000000,1.000000,1.000000,1.000000,1.060000e+02,0.500000
75%,1.000000,0.000000,2.000000,0.000000,1.000000,0.000000,1.000000,0.0,9.000000,1.000000,...,3950.000000,1127.000000,2.000000,5475.250000,11.000000,2.000000,1.000000,1.000000,1.808000e+03,1.000000
max,11.000000,8.000000,29.000000,1.000000,1.000000,1.000000,1.000000,0.0,1541.000000,1.000000,...,7688.000000,7270.000000,3.000000,171421.000000,50.000000,10.000000,1.000000,132.000000,6.108230e+06,1.000000


## Preprocessing

- Replace sentinel value `-1` (no release history available) with column median.
- Impute any remaining NaN with median.
- StandardScale all features before clustering.

In [5]:
X_raw = features_df[PACKAGE_FEATURES].copy()
labels = features_df['label'].values

# -1 is a sentinel for "no release history data" — treat as missing
X_raw.replace(-1, np.nan, inplace=True)

# Impute missing values with column median
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X_raw)

# Standardize for K-Means (sensitive to scale)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

print(f'Feature matrix shape : {X_scaled.shape}')
print(f'NaN remaining        : {np.isnan(X_scaled).sum()}')

missing_pct = X_raw.isna().mean().sort_values(ascending=False)
print('\nMissing % per feature (before imputation):')
print(missing_pct[missing_pct > 0].to_string())

Feature matrix shape : (10000, 20)
NaN remaining        : 0

Missing % per feature (before imputation):
Series([], )


## Step 5: Train K-Means — Elbow Method

Sweep K from 2 to 20. The elbow is detected as the K where the **marginal gain** in inertia reduction drops most sharply (largest second-difference).

In [6]:
K_RANGE = range(2, 21)

inertias = {}
for k in tqdm(K_RANGE, desc='Elbow sweep'):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias[k] = km.inertia_

elbow_df = pd.DataFrame({
    'K'       : list(inertias.keys()),
    'Inertia' : list(inertias.values()),
})
elbow_df['Delta']       = elbow_df['Inertia'].diff().abs()        # 1st difference
elbow_df['Delta2']      = elbow_df['Delta'].diff().abs()          # 2nd difference (elbow signal)
elbow_df['Gain_pct']    = (elbow_df['Delta'] / elbow_df['Inertia'].shift(1) * 100).round(2)

print(elbow_df.to_string(index=False))

# Auto-detect elbow: K where second-difference is maximised
best_k = int(elbow_df.loc[elbow_df['Delta2'].idxmax(), 'K'])
print(f'\nAuto-detected elbow K = {best_k}')

Elbow sweep: 100%|██████████| 19/19 [00:01<00:00,  9.97it/s]

 K       Inertia        Delta      Delta2  Gain_pct
 2 167293.003214          NaN         NaN       NaN
 3 155721.616686 11571.386528         NaN      6.92
 4 145933.042138  9788.574549 1782.811979      6.29
 5 137438.961626  8494.080512 1294.494037      5.82
 6 130961.016347  6477.945279 2016.135233      4.71
 7 122841.626022  8119.390325 1641.445045      6.20
 8 115090.436292  7751.189730  368.200595      6.31
 9 110059.096888  5031.339403 2719.850327      4.37
10 100947.359453  9111.737435 4080.398032      8.28
11  97938.591671  3008.767782 6102.969653      2.98
12  91361.937743  6576.653928 3567.886147      6.72
13  87082.685379  4279.252364 2297.401564      4.68
14  81874.472699  5208.212679  928.960315      5.98
15  78640.481632  3233.991067 1974.221612      3.95
16  74575.917683  4064.563949  830.572882      5.17
17  70464.120176  4111.797506   47.233557      5.51
18  69098.668719  1365.451458 2746.346049      1.94
19  67281.785325  1816.883394  451.431936      2.63
20  64164.92

In [7]:
# Override best_k here if you disagree with the auto-detection
# best_k = 10

final_km = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cluster_labels = final_km.fit_predict(X_scaled)

features_df['cluster'] = cluster_labels
print(f'Trained KMeans with K={best_k}')
print(features_df.groupby('cluster')['label'].agg(['sum', 'count']).rename(
    columns={'sum': 'n_vulnerable', 'count': 'cluster_size'}
))

Trained KMeans with K=11
         n_vulnerable  cluster_size
cluster                            
0                 409           412
1                 837           886
2                 223          1315
3                 200           231
4                 368          1269
5                2020          2126
6                 123          1400
7                  11            11
8                  23            23
9                 739          2242
10                 47            85


In [8]:
from sklearn.metrics import silhouette_score

sil = silhouette_score(X_scaled, cluster_labels)
print(f'Silhouette score (K={best_k}, raw metadata features): {sil:.4f}')
print('  > 0.5   → strong structure')
print('  0.2–0.5 → moderate structure')
print('  < 0.2   → weak / overlapping clusters')

Silhouette score (K=11, raw metadata features): 0.1935
  > 0.5   → strong structure
  0.2–0.5 → moderate structure
  < 0.2   → weak / overlapping clusters


## Step 6: Evaluate

**Risk score** per cluster = `n_vulnerable / cluster_size`  
**Top-K clusters** = top 10% of clusters ranked by risk score (configurable via `TOP_K_PCT`)

Metrics:
- **Recall** — what fraction of all known vulnerable packages fall in the top-K clusters?
- **Precision** — of everything inside the top-K clusters, what fraction is actually vulnerable?

In [17]:
TOP_K_PCT = 0.1

# --- Compute risk score per cluster ---
cluster_stats = (
    features_df.groupby('cluster')['label']
    .agg(n_vulnerable='sum', cluster_size='count')
    .assign(risk_score=lambda df: df['n_vulnerable'] / df['cluster_size'])
    .sort_values('risk_score', ascending=False)
    .reset_index()
)

print('=== All clusters ranked by risk score ===')
print(cluster_stats.to_string(index=False))

# Top-K% of PACKAGES: add clusters in decreasing risk order until ~K% of packages are flagged
n_top_pkgs = max(1, round(len(features_df) * TOP_K_PCT))
top_clusters = set()
n_flagged = 0
for _, row in cluster_stats.iterrows():
    top_clusters.add(int(row['cluster']))
    n_flagged += int(row['cluster_size'])
    if n_flagged >= n_top_pkgs:
        break
print(f'\nTop clusters covering ~{TOP_K_PCT*100:.0f}% of packages ({n_flagged} / {len(features_df)}): {top_clusters}')

# --- Recall & Precision ---
total_vulnerable  = (features_df['label'] == 1).sum()
in_top_mask       = features_df['cluster'].isin(top_clusters)

true_positives    = ((features_df['label'] == 1) & in_top_mask).sum()
top_cluster_size  = in_top_mask.sum()
# print(f"Total_vulnerables: {total_vulnerable}")
# print(f"True positives: {true_positives}")

recall    = true_positives / total_vulnerable if total_vulnerable else 0
precision = true_positives / top_cluster_size if top_cluster_size else 0
f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0

print(f'\n--- Evaluation (top ~{TOP_K_PCT*100:.0f}% packages as "risky") ---')
print(f'Total packages          : {len(features_df)}')
print(f'Total vulnerable        : {total_vulnerable}')
print(f'Flagged packages        : {top_cluster_size}')
print(f'True positives          : {true_positives}')
print(f'Recall                  : {recall:.4f}  ({recall*100:.1f}%)')
print(f'Precision               : {precision:.4f}  ({precision*100:.1f}%)')
print(f'F1                      : {f1:.4f}')

=== All clusters ranked by risk score ===
 cluster  n_vulnerable  cluster_size  risk_score
       7            11            11    1.000000
       8            23            23    1.000000
       0           409           412    0.992718
       5          2020          2126    0.950141
       1           837           886    0.944695
       3           200           231    0.865801
      10            47            85    0.552941
       9           739          2242    0.329616
       4           368          1269    0.289992
       2           223          1315    0.169582
       6           123          1400    0.087857

Top clusters covering ~10% of packages (2572 / 10000): {8, 0, 5, 7}

--- Evaluation (top ~10% packages as "risky") ---
Total packages          : 10000
Total vulnerable        : 5000
Flagged packages        : 2572
True positives          : 2463
Recall                  : 0.4926  (49.3%)
Precision               : 0.9576  (95.8%)
F1                      : 0.6506


## Step 7: Save the baseline model

In [10]:
joblib.dump(final_km, MODEL_OUTPUT)
joblib.dump(scaler,   SCALER_OUTPUT)
joblib.dump(imputer,  IMPUTER_OUTPUT)

# Save cluster risk-score table for downstream use
cluster_stats.to_csv('../models/lib/cluster_risk_scores.csv', index=False)

print(f'Model   saved → {MODEL_OUTPUT}')
print(f'Scaler  saved → {SCALER_OUTPUT}')
print(f'Imputer saved → {IMPUTER_OUTPUT}')
print(f'Cluster risk scores saved → ../models/lib/cluster_risk_scores.csv')

Model   saved → ../models/lib/baseline_kmeans.pkl
Scaler  saved → ../models/lib/baseline_scaler.pkl
Imputer saved → ../models/lib/baseline_imputer.pkl
Cluster risk scores saved → ../models/lib/cluster_risk_scores.csv
